# V27 POC — Tree cover over 20 percent

This notebook evaluates the proposed `TREE_COVER_HIGH` rule against the ignored
real Kobo export in `notebooks/data`, using **ESA WorldCover 2021 v200** — the
static source recommended by V27 D-2 and D-5. Tree cover is measured as the
share of each plot's valid classified area that falls in WorldCover class `10`
(Tree cover).

Privacy rules follow V24: raw rows, names, phone numbers, attachment URLs, and
exact plot coordinates are never displayed, identifiers in outputs are one-way
hashes, and every generated artifact stays below the fully ignored
`notebooks/data/v27_poc_output` directory. The notebook performs no database
writes.

## Scope of this POC

WorldCover tiles are read directly from the public ESA bucket as cloud-optimised
GeoTIFFs, so only the windows covering the collection area are transferred. This
validates the **static (Option A) measurement, threshold, and flag contract**.

The Dynamic World refresh option (V27 D-3) is **not** exercised here: it needs
an Earth Engine project, credentials, and an export pipeline that do not yet
exist for this deployment. That option therefore remains estimated work with no
POC evidence behind it, and Option A remains the recommended v1.

## 1. Environment and configuration

Run from the repository root. Required packages are `numpy`, `rasterio`,
`shapely`, and `pyproj`. GDAL must be able to read `/vsicurl/` URLs; no ESA
account or API key is required. The first run downloads only the windows it
needs and caches them as a local subset, so a rerun performs no network
request.

In [ ]:
from pathlib import Path
import json
import math
import os
import statistics
import sys

import numpy as np
import rasterio
from rasterio.merge import merge
from shapely.geometry import box

sys.path.insert(0, str(Path.cwd() / 'notebooks'))
from poc_common import (
    dataset_utm_crs,
    load_backend_helpers,
    load_records,
    poc_paths,
    project_polygon,
    utm_transformer,
    write_public_output,
)

os.environ.setdefault('GDAL_DISABLE_READDIR_ON_OPEN', 'EMPTY_DIR')
os.environ.setdefault('CPL_VSIL_CURL_ALLOWED_EXTENSIONS', '.tif')

PATHS = poc_paths('v27')
OUTPUT_DIR = PATHS['output_dir']
SUBSET_PATH = OUTPUT_DIR / 'worldcover_2021_subset.tif'
RESULT_PATH = OUTPUT_DIR / 'tree_cover_results_private.json'
AC_DEMO_PATH = OUTPUT_DIR / 'v27_acceptance_demo.json'

WORLDCOVER_BASE = (
    'https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map'
)
LANDCOVER_SOURCE = 'ESA WorldCover'
LANDCOVER_SOURCE_VERSION = '2021-v200'
LANDCOVER_SOURCE_PROVIDER = 'ESA WorldCover / Terrascope'
TREE_COVER_MAX_PERCENT = 20.0
TREE_COVER_RULE_VERSION = 'v1'
MIN_RASTER_COVERAGE_PERCENT = 90.0
PIXEL_RESOLUTION_M = 10
PIXEL_AREA_M2 = 100.0
LOW_CONFIDENCE_PIXEL_EQUIVALENT = 4.0
BBOX_PADDING_DEGREES = 0.005
WORLDCOVER_RESOLUTION_DEG = 1.0 / 12000.0
WORLDCOVER_NODATA = 0

WORLDCOVER_CLASSES = {
    10: 'Tree cover',
    20: 'Shrubland',
    30: 'Grassland',
    40: 'Cropland',
    50: 'Built-up',
    60: 'Bare / sparse vegetation',
    70: 'Snow and ice',
    80: 'Permanent water bodies',
    90: 'Herbaceous wetland',
    95: 'Mangroves',
    100: 'Moss and lichen',
}
TREE_CLASS = 10

print({
    'csv_present': PATHS['csv'].exists(),
    'cached_subset_present': SUBSET_PATH.exists(),
    'threshold_percent': TREE_COVER_MAX_PERCENT,
    'tree_class': TREE_CLASS,
})

## 2. Load the real submissions through the production parser

In [ ]:
HELPERS = load_backend_helpers(PATHS['repo_root'])
records, valid_records = load_records(PATHS['csv'], HELPERS)

print({
    'submission_count': len(records),
    'valid_polygon_count': len(valid_records),
    'invalid_polygon_count': len(records) - len(valid_records),
    'raw_values_displayed': False,
})

## 3. Build a cached WorldCover subset for the collection area

WorldCover is published as 3° × 3° tiles named after their south-west corner.
The collection area can straddle a tile edge, so the notebook resolves every
tile the padded bounding box touches and mosaics only the overlapping windows.
Bounds are deliberately not printed.

`rasterio.merge` with explicit `bounds` and `res` performs range requests
against the remote cloud-optimised GeoTIFFs; the whole 36000 × 36000 tile is
never downloaded. The result is written once to the ignored output directory
and reused afterwards, which is the file contract the production import would
produce.

In [ ]:
all_coords = [coord for record in valid_records for coord in record['coords']]
private_bbox = HELPERS['compute_bbox'](all_coords)

padded = {
    'west': private_bbox['min_lon'] - BBOX_PADDING_DEGREES,
    'east': private_bbox['max_lon'] + BBOX_PADDING_DEGREES,
    'south': private_bbox['min_lat'] - BBOX_PADDING_DEGREES,
    'north': private_bbox['max_lat'] + BBOX_PADDING_DEGREES,
}


def tile_name(lon_corner, lat_corner):
    hemisphere = 'N' if lat_corner >= 0 else 'S'
    meridian = 'E' if lon_corner >= 0 else 'W'
    return (
        f'{hemisphere}{abs(lat_corner):02d}{meridian}{abs(lon_corner):03d}'
    )


def required_tile_urls(bounds):
    lon_start = math.floor(bounds['west'] / 3) * 3
    lat_start = math.floor(bounds['south'] / 3) * 3
    urls = []
    lon = lon_start
    while lon <= bounds['east']:
        lat = lat_start
        while lat <= bounds['north']:
            name = tile_name(lon, lat)
            urls.append(
                f'/vsicurl/{WORLDCOVER_BASE}/'
                f'ESA_WorldCover_10m_2021_v200_{name}_Map.tif'
            )
            lat += 3
        lon += 3
    return urls


def snap_down(value):
    return math.floor(value / WORLDCOVER_RESOLUTION_DEG) * (
        WORLDCOVER_RESOLUTION_DEG
    )


def snap_up(value):
    return math.ceil(value / WORLDCOVER_RESOLUTION_DEG) * (
        WORLDCOVER_RESOLUTION_DEG
    )


def ensure_subset(destination):
    if destination.exists() and destination.stat().st_size > 0:
        return 'cached', []
    sources = []
    opened_names = []
    for url in required_tile_urls(padded):
        try:
            sources.append(rasterio.open(url))
            opened_names.append(url.rsplit('_', 2)[-2])
        except rasterio.errors.RasterioIOError:
            # Tiles over open water are not published.
            continue
    if not sources:
        raise RuntimeError('No WorldCover tile covers the collection area')
    try:
        mosaic, transform = merge(
            sources,
            bounds=(
                snap_down(padded['west']),
                snap_down(padded['south']),
                snap_up(padded['east']),
                snap_up(padded['north']),
            ),
            res=WORLDCOVER_RESOLUTION_DEG,
            nodata=WORLDCOVER_NODATA,
        )
        profile = sources[0].profile.copy()
    finally:
        for source in sources:
            source.close()
    profile.update(
        driver='GTiff',
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        transform=transform,
        nodata=WORLDCOVER_NODATA,
        compress='deflate',
        tiled=False,
        blockxsize=None,
        blockysize=None,
    )
    profile.pop('blockxsize', None)
    profile.pop('blockysize', None)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(destination, 'w', **profile) as sink:
        sink.write(mosaic)
    return 'downloaded', opened_names


subset_state, tiles_used = ensure_subset(SUBSET_PATH)
with rasterio.open(SUBSET_PATH) as landcover:
    LANDCOVER_NODATA = landcover.nodata
    print({
        'subset_state': subset_state,
        'tiles_used': tiles_used,
        'crs': str(landcover.crs),
        'shape': landcover.shape,
        'nodata': LANDCOVER_NODATA,
        'dtype': landcover.dtypes[0],
        'bounds_printed': False,
    })

## 4. Area-weighted tree cover per plot

For every plot the notebook builds the exact intersection between the plot and
each 10 m classified cell and accumulates the intersection area per class,
measured in the collection area's UTM zone. This is what the technical
acceptance criterion "edge pixels are area-weighted" requires: a cell that
covers a tenth of the plot must not count as much as one fully inside it.

The denominator is the **valid classified area** inside the plot, not the plot
area, so nodata never silently inflates or deflates the percentage. A plot
whose valid coverage falls below `MIN_RASTER_COVERAGE_PERCENT` is recorded as
`unavailable` rather than as a pass.

In [ ]:
def tree_cover_flag(percent):
    if percent <= TREE_COVER_MAX_PERCENT:
        return None
    return {
        'type': 'TREE_COVER_HIGH',
        'severity': 'warning',
        'note': (
            f'Tree cover is {percent:.2f}% '
            f'(threshold: > {TREE_COVER_MAX_PERCENT:g}%). '
            f'Dataset source: {LANDCOVER_SOURCE} '
            f'{LANDCOVER_SOURCE_VERSION}.'
        ),
    }


UTM_CRS = dataset_utm_crs(valid_records)
TO_UTM = utm_transformer(UTM_CRS)


def intersecting_cells(raster, polygon_wgs84, pad=1):
    """Yield (row, col, cell polygon) for every cell touching a plot."""
    min_x, min_y, max_x, max_y = polygon_wgs84.bounds
    top_row, left_col = raster.index(min_x, max_y, op=math.floor)
    bottom_row, right_col = raster.index(max_x, min_y, op=math.ceil)
    top_row = max(top_row - pad, 0)
    left_col = max(left_col - pad, 0)
    bottom_row = min(bottom_row + pad, raster.height - 1)
    right_col = min(right_col + pad, raster.width - 1)
    window = rasterio.windows.Window(
        left_col,
        top_row,
        right_col - left_col + 1,
        bottom_row - top_row + 1,
    )
    data = raster.read(1, window=window)
    transform = raster.window_transform(window)
    for row in range(data.shape[0]):
        for col in range(data.shape[1]):
            x0, y0 = transform * (col, row)
            x1, y1 = transform * (col + 1, row + 1)
            cell = box(min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1))
            if cell.intersects(polygon_wgs84):
                yield int(data[row, col]), cell


def measure_tree_cover(polygon_wgs84, raster):
    """Return tree-cover percent and quality details for one plot."""
    plot_utm = project_polygon(polygon_wgs84, TO_UTM)
    plot_area_m2 = plot_utm.area
    area_by_class = {}
    valid_area_m2 = 0.0
    for value, cell in intersecting_cells(raster, polygon_wgs84):
        piece = project_polygon(cell, TO_UTM).intersection(plot_utm)
        if piece.is_empty or piece.area <= 0:
            continue
        if value == raster.nodata:
            continue
        area_by_class[value] = area_by_class.get(value, 0.0) + piece.area
        valid_area_m2 += piece.area

    coverage_percent = (
        valid_area_m2 / plot_area_m2 * 100.0 if plot_area_m2 else 0.0
    )
    tree_area_m2 = area_by_class.get(TREE_CLASS, 0.0)
    effective_pixels = plot_area_m2 / PIXEL_AREA_M2
    details = {
        'tree_area_m2': round(tree_area_m2, 2),
        'valid_covered_area_m2': round(valid_area_m2, 2),
        'plot_area_m2': round(plot_area_m2, 2),
        'coverage_percent': round(coverage_percent, 3),
        'pixel_resolution_m': PIXEL_RESOLUTION_M,
        'effective_pixel_equivalent': round(effective_pixels, 3),
        'resolution_confidence': (
            'low'
            if effective_pixels < LOW_CONFIDENCE_PIXEL_EQUIVALENT
            else 'normal'
        ),
        'class_area_m2': {
            str(class_value): round(area, 2)
            for class_value, area in sorted(area_by_class.items())
        },
    }
    if not valid_area_m2 or coverage_percent < MIN_RASTER_COVERAGE_PERCENT:
        return None, details
    return tree_area_m2 / valid_area_m2 * 100.0, details


results = []
with rasterio.open(SUBSET_PATH) as landcover:
    for record in valid_records:
        percent, details = measure_tree_cover(record['polygon'], landcover)
        if percent is None:
            results.append({
                'plot_ref': record['plot_ref'],
                'status': 'unavailable',
                'value': None,
                'details': details,
                'flag': None,
            })
            continue
        results.append({
            'plot_ref': record['plot_ref'],
            'status': 'complete',
            'value': round(percent, 3),
            'details': details,
            'flag': tree_cover_flag(percent),
        })

complete_results = [r for r in results if r['status'] == 'complete']
print({
    'analysed_plots': len(results),
    'complete': len(complete_results),
    'unavailable': len(results) - len(complete_results),
})

## 5. Aggregate outcome and observed land-cover mix

The class breakdown matters as much as the flag count. If the collection area
is dominated by a single non-tree class, the 20% rule is cheap to run but rarely
discriminates; if tree and shrub classes are adjacent at 10 m resolution, the
rule is sensitive to how WorldCover drew that boundary rather than to the farm
itself.

In [ ]:
values = [r['value'] for r in complete_results]
flagged = [r for r in complete_results if r['flag']]
low_confidence = [
    r for r in complete_results
    if r['details']['resolution_confidence'] == 'low'
]
zero_tree = [r for r in complete_results if r['value'] == 0.0]

total_class_area = {}
for result in complete_results:
    for class_value, area in result['details']['class_area_m2'].items():
        total_class_area[class_value] = (
            total_class_area.get(class_value, 0.0) + area
        )
total_area = sum(total_class_area.values())

summary = {
    'valid_input_polygons': len(valid_records),
    'completed_calculations': len(complete_results),
    'flagged_over_20_percent': len(flagged),
    'plots_with_zero_tree_cover': len(zero_tree),
    'min_tree_cover_percent': round(min(values), 2),
    'median_tree_cover_percent': round(statistics.median(values), 2),
    'max_tree_cover_percent': round(max(values), 2),
    'min_coverage_percent': round(
        min(r['details']['coverage_percent'] for r in complete_results), 2
    ),
    'low_resolution_confidence_plots': len(low_confidence),
    'median_effective_pixel_equivalent': round(
        statistics.median(
            r['details']['effective_pixel_equivalent']
            for r in complete_results
        ),
        3,
    ),
    'observed_class_share_percent': {
        WORLDCOVER_CLASSES.get(int(class_value), class_value): round(
            area / total_area * 100.0, 2
        )
        for class_value, area in sorted(
            total_class_area.items(), key=lambda item: -item[1]
        )
    },
}
print(json.dumps(summary, indent=2))

## 6. Boundary-contract demo using existing records

The rule is strict: exactly `20.00%` is not flagged and `20.01%` is. Each
scenario reuses an existing hashed plot record and its real measured value, but
substitutes a controlled evaluation value. Controlled values are labelled and
never replace the observed measurement.

In [ ]:
if len(complete_results) < 3:
    raise RuntimeError('At least three complete results are required')

controlled_scenarios = [
    ('below_threshold', complete_results[0], 19.99),
    ('exactly_at_threshold', complete_results[1], 20.00),
    ('above_threshold', complete_results[2], 20.01),
]

acceptance_rows = []
for scenario, real_result, controlled_value in controlled_scenarios:
    flag = tree_cover_flag(controlled_value)
    acceptance_rows.append({
        'scenario': scenario,
        'uses_existing_hashed_plot': True,
        'plot_ref': real_result['plot_ref'],
        'observed_tree_cover_percent': real_result['value'],
        'controlled_demo_tree_cover_percent': controlled_value,
        'controlled_demo_input': True,
        'flagged_for_review': bool(flag),
        'flagged_reason': [flag] if flag else [],
        'geospatial_metrics': {
            'tree_cover': {
                'status': 'complete',
                'value': controlled_value,
                'unit': 'percent',
                'source': LANDCOVER_SOURCE,
                'source_version': LANDCOVER_SOURCE_VERSION,
                'source_provider': LANDCOVER_SOURCE_PROVIDER,
                'threshold': {
                    'operator': '>',
                    'value': TREE_COVER_MAX_PERCENT,
                    'unit': 'percent',
                    'rule_version': TREE_COVER_RULE_VERSION,
                },
                'details': real_result['details'],
            }
        },
    })

AC_DEMO_PATH.write_text(json.dumps(acceptance_rows, indent=2))
print([
    (row['scenario'], row['flagged_for_review'])
    for row in acceptance_rows
])

## 7. Resolution sensitivity — what one pixel is worth

At 10 m a single WorldCover cell is 100 m². The check below reports, per plot,
how much the result would move if exactly one boundary cell were reclassified
between tree and non-tree. Where that swing is larger than the distance from
the measured value to the 20% threshold, the outcome of the rule depends on a
single pixel classification and the stored result must say so.

In [ ]:
single_pixel_swings = []
decided_by_one_pixel = 0
for result in complete_results:
    valid_area = result['details']['valid_covered_area_m2']
    if not valid_area:
        continue
    swing_percent = PIXEL_AREA_M2 / valid_area * 100.0
    single_pixel_swings.append(swing_percent)
    if abs(result['value'] - TREE_COVER_MAX_PERCENT) < swing_percent:
        decided_by_one_pixel += 1

print(json.dumps({
    'median_single_pixel_swing_percent': round(
        statistics.median(single_pixel_swings), 2
    ),
    'max_single_pixel_swing_percent': round(max(single_pixel_swings), 2),
    'plots_where_one_pixel_decides_the_flag': decided_by_one_pixel,
    'plots_evaluated': len(single_pixel_swings),
}, indent=2))

## 8. Persist the POC artifacts

Output is split by sensitivity.

`notebooks/outputs/` is **tracked by git** and receives the aggregate summary,
the observed land-cover mix, the single-pixel sensitivity result, and the
boundary demo.

`notebooks/data/v27_poc_output/` stays **fully ignored**. It holds the per-plot
rows and `worldcover_2021_subset.tif`, whose georeferenced bounds are the padded
plot bounding box — opening it in QGIS discloses the collection area even though
WorldCover itself is public.

In [ ]:
payload = {
    'summary': summary,
    'dataset': {
        'source': LANDCOVER_SOURCE,
        'source_version': LANDCOVER_SOURCE_VERSION,
        'source_provider': LANDCOVER_SOURCE_PROVIDER,
        'tree_class': TREE_CLASS,
        'pixel_resolution_m': PIXEL_RESOLUTION_M,
        'refresh_option_exercised': False,
    },
    'results': [
        {
            'plot_ref': result['plot_ref'],
            'geospatial_metrics': {
                'tree_cover': {
                    'status': result['status'],
                    'value': result['value'],
                    'unit': 'percent',
                    'source': LANDCOVER_SOURCE,
                    'source_version': LANDCOVER_SOURCE_VERSION,
                    'source_provider': LANDCOVER_SOURCE_PROVIDER,
                    'threshold': {
                        'operator': '>',
                        'value': TREE_COVER_MAX_PERCENT,
                        'unit': 'percent',
                        'rule_version': TREE_COVER_RULE_VERSION,
                    },
                    'details': result['details'],
                }
            },
            'flagged_reason': [result['flag']] if result['flag'] else [],
        }
        for result in results
    ],
}
RESULT_PATH.write_text(json.dumps(payload, indent=2))

public = write_public_output('v27_tree_cover_summary.json', {
    'poc': 'v27',
    'metric': 'tree_cover',
    'notebook': 'notebooks/v27_tree_cover_poc.ipynb',
    'dataset': payload['dataset'],
    'summary': summary,
    'single_pixel_sensitivity': {
        'median_single_pixel_swing_percent': round(
            statistics.median(single_pixel_swings), 2
        ),
        'max_single_pixel_swing_percent': round(max(single_pixel_swings), 2),
        'plots_where_one_pixel_decides_the_flag': decided_by_one_pixel,
        'plots_evaluated': len(single_pixel_swings),
    },
    'boundary_demo': [
        {
            'scenario': row['scenario'],
            'controlled_demo_tree_cover_percent': (
                row['controlled_demo_tree_cover_percent']
            ),
            'flagged_for_review': row['flagged_for_review'],
            'note': (
                row['flagged_reason'][0]['note']
                if row['flagged_reason'] else None
            ),
        }
        for row in acceptance_rows
    ],
})

print({
    'ignored_private_files': [
        RESULT_PATH.name, AC_DEMO_PATH.name, SUBSET_PATH.name,
    ],
    'tracked_public_file': public['file'],
})

## 10. Synthetic example — what a flagged plot looks like

Only 4 of 166 real plots exceed 20%, and 160 sit at exactly 0%, so the positive
branch is barely visible in the real data. This section hand-draws a 25 m
example plot inside a **public forest far from the collection area**, measures
it with `measure_tree_cover` above, and previews it on a basemap.

Two properties make this safe to commit: the polygon is fabricated, and the
location is a named landmark rather than a farm, so the map discloses nothing
about where African Bamboo works. Land cover is read straight from the public
ESA WorldCover COG for that tile, so no cached subset is involved.

In [ ]:
from poc_common import (
    DEMO_LOCATIONS,
    preview_map,
    public_worldcover_url,
    synthetic_square,
)

DEMO_SIDE_M = 25.0
demo_lon, demo_lat, demo_place = DEMO_LOCATIONS['forest']
demo_plot = synthetic_square(demo_lon, demo_lat, DEMO_SIDE_M)

# UTM zone 37N covers 36-42E, so the dataset transformer built above is valid
# at this demo location too.
with rasterio.open(public_worldcover_url(demo_lon, demo_lat)) as demo_lc:
    demo_value, demo_details = measure_tree_cover(demo_plot, demo_lc)

demo_flag = tree_cover_flag(demo_value) if demo_value is not None else None
print(json.dumps({
    'location': demo_place,
    'tree_cover_percent': round(demo_value, 2) if demo_value else None,
    'threshold_percent': TREE_COVER_MAX_PERCENT,
    'flagged': bool(demo_flag),
    'note': demo_flag['note'] if demo_flag else None,
    'class_area_m2': demo_details['class_area_m2'],
    'single_pixel_swing_percent': round(
        PIXEL_AREA_M2 / demo_details['valid_covered_area_m2'] * 100.0, 2
    ),
}, indent=2))

The map below is the visual check. A red outline means the rule
fired; click the polygon for the measured values. Run
`pip install -r notebooks/requirements.txt` if folium is missing.

In [ ]:
preview_map(
    demo_plot,
    flagged=bool(demo_flag),
    title=f'V27 example - {demo_place}',
    rows={
        'Tree cover': f'{demo_value:.2f}%',
        'Threshold': f'> {TREE_COVER_MAX_PERCENT:g}%',
        'Result': 'TREE_COVER_HIGH' if demo_flag else 'pass',
        'Resolution': f"{demo_details['effective_pixel_equivalent']} cells",
        'Source': f'{LANDCOVER_SOURCE} {LANDCOVER_SOURCE_VERSION}',
    },
    zoom=17,
)

## 9. Review checklist and interpretation

The POC is technically successful when every valid geoshape parses, the
WorldCover subset covers all plots, edge cells are area-weighted, nodata is
excluded from the denominator, insufficient coverage is reported as
`unavailable` rather than as a pass, and the strict-boundary demo shows that
exactly `20.00%` is not flagged.

The interpretation questions are for product and GIS, and the printed summary
answers them directly:

1. **Does the rule discriminate here?** `observed_class_share_percent` and
   `flagged_over_20_percent` show whether the collection area contains enough
   tree-classified ground for the rule to separate plots at all.
2. **Is 10 m fine enough?** Section 7 reports how many plots sit closer to the
   threshold than the value of one boundary pixel. For those plots the flag is
   decided by a single WorldCover classification, and Helen's UI must present
   the number with that caveat rather than as a precise measurement.
3. **Static or refreshed?** Nothing in this run argues for the Dynamic World
   pipeline. WorldCover 2021 describes 2021 conditions; recency only becomes
   worth its cost if product confirms that recent land-cover change is part of
   the decision Helen is making.

For an independent check, load the ignored `worldcover_2021_subset.tif` in QGIS
and compare several hashed records with Zonal Statistics or the raster
histogram. Do not copy the real polygons or the raster outside the ignored POC
directory.

Attribution: ESA WorldCover project 2021 / Contains modified Copernicus
Sentinel data (2021) processed by ESA WorldCover consortium, licensed under
CC BY 4.0.